# 🚛 Nestlé VFR — Master Truck Loading Optimizer v9 (Production Edition)
**Universal offline notebook — works in Jupyter, JupyterLab, VS Code, and Miniconda.**
**The optimization algorithm itself is unchanged from v9 — this edition only adds
reliability, logging, and easier deployment around it.**

### Folder layout
```
VFR_Project/
├── VFR_Master_Optimizer_v9_local.ipynb   ← this notebook
├── requirements.txt
├── install.bat        ← double-click once, first time only
├── run.bat             ← double-click every time you want to run it
├── run_auto.bat        ← optional: runs everything with no browser window
├── README.md
├── INPUT/               ← place your Excel input file here
├── OUTPUT/               ← optimized loading plan is saved here automatically
└── LOGS/                  ← run_log.txt — every run's history, for troubleshooting
```

### How to run
1. Drop your Excel input file into the `INPUT/` folder beside this notebook.
2. From the Jupyter/VS Code toolbar choose **Run All** (or run cells 1 → 5 in order).
3. Watch the progress messages: *Checking dependencies… → Reading Excel… →
   Building SKU tables… → Running optimizer… → Generating output… → Done.*
4. Your loading plan appears in `OUTPUT/` as a timestamped `.xlsx` file.

> No file upload dialogs. No Google Drive. Works fully offline.
> If anything goes wrong, a short plain-English message is shown and the
> full technical detail is saved to `LOGS/run_log.txt`.

### Cell map
| Cell | Purpose |
|---|---|
| 1 | Setup — checks/installs dependencies, creates INPUT/OUTPUT/LOGS, sets up logging & friendly error handling |
| 2 | Detects and reads the Excel file in `INPUT/` |
| 3 | Progress marker |
| 4 | **The optimizer + Excel export — identical algorithm to the original v9 notebook, unmodified** |
| 5 | Validates the output file and prints a summary |

Set `DEBUG_MODE = True` at the top of Cell 1 if you ever need to see the full
Python error text instead of the short business-friendly message.

---

### Algorithm overview (unchanged)

**Phase 1 — Non-filler demand fulfillment**
- Non-fillers sorted: 0% tolerance first, then by planned_qty ASC
- For each Non-filler (target):
  - Find best height partner: tries Non-fillers first, then fillers
  - Compares best pair vs best solo — picks solo only if pair gap is >80 mm worse
  - **z is always driven by the target SKU's remaining demand** — partner never inflates z
  - Repeats stacks until that Non-filler hits dmin, then moves to next
- Non-filler tolerance is a hard wall — dmax is never breached

**Phase 2 — Filler demand fulfillment**
- Same logic applied to fillers (smallest planned qty first)
- Partners are any non-blocked SKU

**Phase 3 — Fill remaining truck length**
- Best cartons/mm wins — solo or pair, whoever scores highest

**Phase 3 ⟳ Terminal Residual Rotation Recovery**
- After the main fill loop: if len_rem > 0, tries rotating filler SKUs to recover residual depth

### Pass colours in output
- 🔵 Blue = P1 Non-filler demand
- 🟢 Green = P2 Filler demand
- 🩵 Teal = P3 Fill remaining space
- 🔄 Amber = Rotated orientation


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Setup: dependency check, folders, logging, error handling
# (Deployment/robustness wrapper only. Does NOT touch optimizer logic.)
# ─────────────────────────────────────────────────────────────────────────────
import sys, subprocess, importlib, traceback, datetime, math, warnings
from pathlib import Path
from itertools import combinations, permutations
warnings.filterwarnings('ignore')

DEBUG_MODE = False   # ← set to True to see full technical Python tracebacks

print("Checking dependencies...")

# ── Project folder layout (relative to this notebook; works anywhere) ────────
try:
    PROJECT_ROOT = Path(__file__).resolve().parent   # when run as a script
except NameError:
    PROJECT_ROOT = Path.cwd()                          # Jupyter / VS Code

INPUT_DIR  = PROJECT_ROOT / "INPUT"
OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"
LOGS_DIR   = PROJECT_ROOT / "LOGS"
for _d in (INPUT_DIR, OUTPUT_DIR, LOGS_DIR):
    _d.mkdir(parents=True, exist_ok=True)   # never crash if a folder is missing

# ── Logging (never allowed to crash the run) ──────────────────────────────────
LOG_FILE = LOGS_DIR / "run_log.txt"
_run_started = datetime.datetime.now()

def log(msg, level="INFO"):
    """Append a timestamped line to LOGS/run_log.txt. Silently no-ops on failure."""
    line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] [{level}] {msg}"
    try:
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception:
        pass

log("=== Run started ===")
log(f"Project root : {PROJECT_ROOT}")
log(f"Python       : {sys.executable} ({sys.version.split()[0]})")

# ── Dependency check + automatic install (uses sys.executable — no assumptions
#    about which Python is on PATH; safe to re-run, never installs twice) ─────
REQUIRED_PACKAGES = {
    "pandas":   "pandas",
    "numpy":    "numpy",
    "openpyxl": "openpyxl",
}

def ensure_package(import_name, pip_name):
    try:
        importlib.import_module(import_name)
        print(f"  ✅ {pip_name} already installed")
        return
    except ImportError:
        pass
    print(f"  ⬇️  Installing {pip_name} ...")
    log(f"Installing missing package: {pip_name}")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", pip_name, "-q"]
        )
        importlib.invalidate_caches()
        importlib.import_module(import_name)
        print(f"  ✅ {pip_name} installed")
    except Exception as e:
        log(f"FAILED to install {pip_name}: {e}", level="ERROR")
        raise RuntimeError(
            f"Could not automatically install '{pip_name}'.\n"
            f"   This can happen on locked-down corporate laptops where automatic\n"
            f"   installation is blocked by IT policy — that part cannot be fully\n"
            f"   automated in that case.\n"
            f"   Workaround: ask IT to run this one line in Command Prompt:\n"
            f'       "{sys.executable}" -m pip install {pip_name}\n'
            f"   Then re-run this cell."
        )

for _imp, _pip in REQUIRED_PACKAGES.items():
    ensure_package(_imp, _pip)

import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
print("✅ All dependencies ready.\n")

# ── Notebook-wide friendly error handling ─────────────────────────────────────
# Registered once here so that ANY later cell — including the optimizer cell,
# which is left 100% untouched — shows a short business-friendly message
# instead of a raw Python stack trace. The full technical trace is always
# written to LOGS/run_log.txt so nothing is ever lost.
def _friendly_exception_handler(shell, etype, evalue, tb, tb_offset=None):
    full_trace = "".join(traceback.format_exception(etype, evalue, tb))
    log(f"EXCEPTION: {etype.__name__}: {evalue}\n{full_trace}", level="ERROR")
    print("\n❌ Something went wrong and the run could not finish.")
    print(f"   Reason: {evalue}")
    print(f"   Details were saved to: {LOG_FILE}")
    print("   Please share that file with the analytics/IT team if you need help.")
    if DEBUG_MODE:
        print("\n--- DEBUG MODE: full technical details ---")
        print(full_trace)

try:
    ip = get_ipython()  # noqa: F821  (only defined inside Jupyter/IPython)
    ip.set_custom_exc((Exception,), _friendly_exception_handler)
except NameError:
    pass  # running as a plain .py script — normal Python errors still apply

print(f"Project root : {PROJECT_ROOT}")
print(f"INPUT  folder: {INPUT_DIR}")
print(f"OUTPUT folder: {OUTPUT_DIR}")
print(f"LOGS   folder: {LOGS_DIR}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Auto-detect + read Excel input from INPUT/ folder
# (Parsing logic identical to the original notebook — only wrapped with
#  friendly error handling, logging, and multi-file selection.)
# ─────────────────────────────────────────────────────────────────────────────
print("Reading Excel...")
log("Scanning INPUT/ folder for Excel files")

try:
    # Ignore Excel's temporary "~$file.xlsx" lock files
    xlsx_files = sorted(
        [p for p in INPUT_DIR.glob("*.xlsx") if not p.name.startswith("~$")],
        key=lambda p: p.stat().st_mtime, reverse=True,
    )

    if not xlsx_files:
        friendly = (
            f"No Excel file found in the INPUT folder.\n"
            f"   → Please copy your input .xlsx file into:\n"
            f"     {INPUT_DIR}\n"
            f"   → Then re-run this cell."
        )
        log("No input files found in INPUT/", level="WARNING")
        print(f"\n⚠️  {friendly}")
        raise FileNotFoundError(friendly)

    if len(xlsx_files) == 1:
        input_path = xlsx_files[0]
        print(f"📁 Input file: {input_path.name}")
    else:
        print("⚠️  Multiple Excel files found in INPUT/:")
        for i, p in enumerate(xlsx_files, 1):
            print(f"   [{i}] {p.name}")
        input_path = xlsx_files[0]  # safe default = most recently modified
        try:
            choice = input(
                f"Type a number [1-{len(xlsx_files)}] and press Enter "
                f"(or just press Enter to use the most recent: {input_path.name}): "
            ).strip()
            if choice:
                idx = int(choice) - 1
                if 0 <= idx < len(xlsx_files):
                    input_path = xlsx_files[idx]
        except Exception:
            # "Run All" / headless execution has no console to type into —
            # fall back to the most recent file instead of crashing.
            print("   (Non-interactive run — defaulting to most recent file.)")
        print(f"✅ Using: {input_path.name}")

    log(f"Selected input file: {input_path.name}")

    # ── Read truck dimensions (unchanged logic) ──────────────────────────────
    xl = pd.read_excel(input_path, sheet_name='Inputs', header=None)
    truck = {
        'H':      float(xl.iloc[3, 2]),
        'W':      float(xl.iloc[4, 2]),
        'L':      float(xl.iloc[5, 2]),
        'max_wt': float(xl.iloc[6, 2]),
    }

    print("Building SKU tables...")

    # ── Read SKU data (unchanged logic) ───────────────────────────────────────
    raw = pd.read_excel(input_path, sheet_name='Inputs', header=9,
                        usecols=[0,1,2,3,4,5,6,7,8,9,10])
    raw.columns = ['sku_code','sku_name','planned_qty',
                   'height_mm','width_mm','length_mm',
                   'weight_per_carton','volume_m3',
                   'fulfillment_type','tolerance','sku_type']
    raw = raw[pd.to_numeric(raw['sku_code'], errors='coerce').notna()].reset_index(drop=True)
    for col in ['planned_qty','height_mm','width_mm','length_mm','weight_per_carton','tolerance']:
        raw[col] = pd.to_numeric(raw[col], errors='coerce')
    raw['sku_code']         = raw['sku_code'].astype(int).astype(str)
    raw['planned_qty']      = raw['planned_qty'].astype(int)
    raw['sku_type']         = raw['sku_type'].str.strip()
    raw['fulfillment_type'] = raw['fulfillment_type'].str.strip()
    raw['demand_min']       = (raw['planned_qty'] * (1 - raw['tolerance'])).apply(math.floor)
    raw['demand_max']       = (raw['planned_qty'] * (1 + raw['tolerance'])).apply(math.ceil)

    if raw.empty:
        raise ValueError(
            "No valid SKU rows were found in the 'Inputs' sheet.\n"
            "   → Please check the file matches the expected template "
            "(same sheet name and column layout as before)."
        )

    print(f"\n📦 TRUCK: {truck}")
    print(f"\n📋 {len(raw)} SKUs | {raw['planned_qty'].sum()} cartons planned")
    print(raw[['sku_code','sku_name','planned_qty','height_mm','width_mm',
               'length_mm','weight_per_carton','fulfillment_type',
               'tolerance','sku_type','demand_min','demand_max']].to_string(index=False))

    log(f"Loaded {len(raw)} SKUs from '{input_path.name}'. Truck: {truck}")

except FileNotFoundError:
    raise
except Exception as e:
    log(f"Failed while reading input Excel: {e}", level="ERROR")
    raise RuntimeError(
        "Could not read the input Excel file.\n"
        "   → Please check it matches the expected template "
        "(sheet 'Inputs', same column layout as before).\n"
        f"   → Technical detail: {e}"
    )


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Run optimizer (progress marker only — the actual engine is
# the next cell, kept byte-for-byte identical to the original v9 notebook)
# ─────────────────────────────────────────────────────────────────────────────
print("Running optimizer...")
log("Starting optimization engine (v9 core — logic unmodified)")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Optimize + Export  ── v9 (local)
#
# ALGORITHM:
#  Phase 1 — Non-filler demand fulfillment (strict priority):
#    • Sort Non-fillers: tol=0% first, then by planned_qty ASC
#    • For each Non-filler (target):
#        - Find best partner: try Non-fillers first (gap_mm minimised),
#          then fillers, then solo — whichever gives smallest residual gap
#        - Compare best-pair gap vs best-solo gap; pick solo if gap is
#          materially better (GAP_SOLO_PREFER_MM threshold)
#        - z = ceil(target_remaining / cpz_target) — target drives z, NOT partner
#        - Place stack; target is now exhausted (blocked at dmax)
#        - Move to next Non-filler
#  Phase 2 — Filler demand fulfillment:
#    • Pair remaining fillers with best available partner (Non-filler or filler)
#    • Same logic: smaller filler drives z
#  Phase 3 — Fill remaining truck length:
#    • Best cartons/mm combos until length exhausted
#  Phase 3 ⟳ Terminal Residual Rotation Recovery:
#    • Tries rotating filler SKUs to fill any residual depth after main fill
#
# TOLERANCE:
#    • Non-filler: hard wall at dmax — never exceeded
#    • Filler: soft — loaded up to dmax in fill phase
# ─────────────────────────────────────────────────────────────────────────────

import datetime  # already imported in Cell 1; safe to re-import

LENGTH_BUDGET       = truck['L'] * 0.99
WEIGHT_LIMIT        = truck['max_wt']
MAX_STACKS          = 30
GAP_SOLO_PREFER_MM  = 80   # prefer solo over pair if pair gap exceeds solo gap by this much

# ── cpw: try both orientations, pick the one fitting more cartons across width ─
def best_cpw_dpz(s, tw):
    """Returns (cpw, dpz) — orientation giving max cpw, dpz is the depth dimension."""
    opt1 = (int(tw // s['width_mm']),  s['length_mm'])   # width across, length is depth
    opt2 = (int(tw // s['length_mm']), s['width_mm'])     # length across, width is depth
    # pick higher cpw; tie-break: smaller dpz (uses less truck length)
    if opt1[0] > opt2[0]: return opt1
    if opt2[0] > opt1[0]: return opt2
    return opt1 if opt1[1] <= opt2[1] else opt2  # tie: smaller depth

# Pre-compute per-SKU
lookup = raw.set_index('sku_code').to_dict('index')
for code, s in lookup.items():
    cpw, dpz     = best_cpw_dpz(s, truck['W'])
    s['cpw']     = cpw
    s['dpz']     = dpz
    s['maxL']    = int(truck['H'] // s['height_mm'])

# ── State ─────────────────────────────────────────────────────────────────────
planned  = dict(zip(raw['sku_code'], raw['planned_qty'].astype(int)))
dmin     = dict(zip(raw['sku_code'], raw['demand_min'].astype(int)))
dmax     = dict(zip(raw['sku_code'], raw['demand_max'].astype(int)))
stype    = dict(zip(raw['sku_code'], raw['sku_type']))

loaded   = {c: 0 for c in planned}
blocked  = set()
len_rem  = LENGTH_BUDGET
wt_load  = 0.0
stacks   = []

def remaining(code):   return max(0, dmin[code] - loaded[code])
def is_done(code):     return loaded[code] >= dmin[code]
def is_blocked(code):  return code in blocked or loaded[code] >= dmax[code]

def mark_blocked():
    for c in planned:
        if loaded[c] >= dmax[c]:
            blocked.add(c)

def can_take_z1(code, cpz):
    """True if adding cpz cartons won't breach dmax."""
    return loaded[code] + cpz <= dmax[code]

# ── All layer configs for a single SKU ───────────────────────────────────────
def solo_layer_configs(code):
    """Returns list of (la, cpz, gap_mm, dpz) sorted by gap_mm ASC."""
    s = lookup[code]
    configs = []
    for la in range(1, s['maxL'] + 1):
        cpz   = la * s['cpw']
        gap   = truck['H'] - la * s['height_mm']
        configs.append((la, cpz, gap, s['dpz']))
    configs.sort(key=lambda x: x[2])   # best gap first
    return configs

# ── All layer configs for a pair of SKUs ─────────────────────────────────────
def pair_layer_configs(ca, cb):
    """
    Returns list of dicts for every (la, lb) layer combo that fits in truck height.
    Sorted by gap_mm ASC (height optimisation secondary criterion).
    """
    a, b    = lookup[ca], lookup[cb]
    dpz     = max(a['dpz'], b['dpz'])
    configs = []
    for la in range(1, a['maxL'] + 1):
        for lb in range(1, b['maxL'] + 1):
            cum_h = la * a['height_mm'] + lb * b['height_mm']
            if cum_h > truck['H']: continue
            gap   = truck['H'] - cum_h
            cpz_a = la * a['cpw']
            cpz_b = lb * b['cpw']
            if cpz_a == 0 or cpz_b == 0: continue
            wt_pz = cpz_a * a['weight_per_carton'] + cpz_b * b['weight_per_carton']
            configs.append({
                'codes':  [ca, cb],
                'names':  [a['sku_name'], b['sku_name']],
                'layers': [la, lb],
                'cpws':   [a['cpw'], b['cpw']],
                'cpzs':   [cpz_a, cpz_b],
                'cum_h':  cum_h, 'gap_mm': gap,
                'dpz':    dpz,
                'wt_pz':  wt_pz,
            })
    configs.sort(key=lambda x: x['gap_mm'])
    return configs

# ── Compute z for a pair where `target` drives the zone count ─────────────────
def calc_z_target_driven(cfg, target_code, partner_code):
    """
    z = ceil(target_remaining / cpz_target).
    Hard-capped by: length budget, weight, dmax of BOTH SKUs.
    Returns 0 if any cap makes z impossible.
    """
    ti   = cfg['codes'].index(target_code)
    pi   = cfg['codes'].index(partner_code)
    cpz_t = cfg['cpzs'][ti]
    cpz_p = cfg['cpzs'][pi]

    need  = remaining(target_code)
    if need <= 0: return 0
    if not can_take_z1(target_code, cpz_t): return 0
    if not can_take_z1(partner_code, cpz_p): return 0

    z_demand = math.ceil(need / cpz_t)                          # target drives z
    z_dmax_t = int((dmax[target_code]  - loaded[target_code])  / cpz_t)
    z_dmax_p = int((dmax[partner_code] - loaded[partner_code]) / cpz_p)
    z_len    = int(len_rem // cfg['dpz'])
    z_wt     = int((WEIGHT_LIMIT - wt_load) / cfg['wt_pz']) if cfg['wt_pz'] > 0 else 9999

    z = min(z_demand, z_dmax_t, z_dmax_p, z_len, z_wt)
    return max(0, z)

# ── Compute z for solo stack ──────────────────────────────────────────────────
def calc_z_solo(code, la, cpz, dpz_val):
    """
    z = ceil(remaining / cpz). Capped by length, weight, dmax.
    """
    need  = remaining(code)
    if need <= 0: return 0
    if not can_take_z1(code, cpz): return 0

    z_demand = math.ceil(need / cpz)
    z_dmax   = int((dmax[code] - loaded[code]) / cpz)
    z_len    = int(len_rem // dpz_val)
    wt_pz    = cpz * lookup[code]['weight_per_carton']
    z_wt     = int((WEIGHT_LIMIT - wt_load) / wt_pz) if wt_pz > 0 else 9999

    z = min(z_demand, z_dmax, z_len, z_wt)
    return max(0, z)

# ── Compute z for fill pass (no demand target — just max cartons) ─────────────
def calc_z_fill(cfg_or_solo, is_solo=False, solo_code=None):
    if is_solo:
        code, la, cpz, dpz_val = solo_code
        if is_blocked(code): return 0
        z_dmax = int((dmax[code] - loaded[code]) / cpz)
        z_len  = int(len_rem // dpz_val)
        wt_pz  = cpz * lookup[code]['weight_per_carton']
        z_wt   = int((WEIGHT_LIMIT - wt_load) / wt_pz) if wt_pz > 0 else 9999
        return max(0, min(z_dmax, z_len, z_wt))
    else:
        cfg = cfg_or_solo
        for code, cpz in zip(cfg['codes'], cfg['cpzs']):
            if is_blocked(code): return 0
            if not can_take_z1(code, cpz): return 0
        z_dmax = min(int((dmax[c] - loaded[c]) / cp)
                     for c, cp in zip(cfg['codes'], cfg['cpzs']) if cp > 0)
        z_len  = int(len_rem // cfg['dpz'])
        z_wt   = int((WEIGHT_LIMIT - wt_load) / cfg['wt_pz']) if cfg['wt_pz'] > 0 else 9999
        return max(0, min(z_dmax, z_len, z_wt))

# ── Record and apply a stack ──────────────────────────────────────────────────
def add_stack(codes, names, layers, cpws, cpzs, z, cum_h, gap, dpz_val, wt_pz, label, orientation='Normal'):
    global len_rem, wt_load
    cartons   = [cpz * z for cpz in cpzs]
    lu        = dpz_val * z
    wt        = wt_pz * z
    n         = len(codes)

    for code, ct in zip(codes, cartons):
        loaded[code] += ct
    len_rem -= lu
    wt_load += wt
    mark_blocked()

    row = {
        'stack_id':      f"S{len(stacks)+1}",
        'pass':          label,
        'orientation':   orientation,
        'n_skus':        n,
        'cum_h':         cum_h,
        'gap_mm':        gap,
        'dpz':           dpz_val,
        'z':             z,
        'total_cartons': sum(cartons),
        'length_used':   lu,
        'wt_kg':         round(wt, 1),
    }
    for i in range(3):
        if i < n:
            row[f'code_{i}']    = codes[i]
            row[f'name_{i}']    = names[i]
            row[f'layers_{i}']  = layers[i]
            row[f'cpw_{i}']     = cpws[i]
            row[f'cpz_{i}']     = cpzs[i]
            row[f'cartons_{i}'] = cartons[i]
        else:
            row[f'code_{i}']    = '—'
            row[f'name_{i}']    = ''
            row[f'layers_{i}']  = 0
            row[f'cpw_{i}']     = 0
            row[f'cpz_{i}']     = 0
            row[f'cartons_{i}'] = 0
    stacks.append(row)

    sku_str = ' + '.join(f"{names[i][:18]}(L={layers[i]})" for i in range(n))
    cts_str = '+'.join(str(c) for c in cartons)
    orient_tag = '🔄' if orientation == 'Rotated' else '  '
    print(f"  {orient_tag}[S{len(stacks)}|{label}] {sku_str} | z={z} | {cts_str} cts | "
          f"H:{cum_h:.0f}mm gap:{gap:.0f}mm | len:{lu:.0f}mm rem:{len_rem:.0f}mm"
          f" [{orientation}]")

# ── Place one stack for a target SKU (pair or solo) ──────────────────────────
def place_target(target, candidate_partners, label):
    """
    Find best placement for `target` SKU.
    Tries Non-filler partners first, then filler partners, then solo.
    Picks lowest gap. If solo gap is materially better than best pair, goes solo.
    Drives z from target's remaining demand.
    Returns True if a stack was placed.
    """
    s_t = lookup[target]

    # ── Best solo option ──────────────────────────────────────────────────────
    best_solo      = None
    best_solo_gap  = 9999
    best_solo_z    = 0
    for la, cpz, gap, dpz_val in solo_layer_configs(target):
        z = calc_z_solo(target, la, cpz, dpz_val)
        if z < 1: continue
        if gap < best_solo_gap:
            best_solo_gap = gap
            best_solo     = (la, cpz, dpz_val)
            best_solo_z   = z
        break   # solo_layer_configs sorted by gap ASC — first valid is best

    # ── Best pair option ──────────────────────────────────────────────────────
    best_pair     = None
    best_pair_gap = 9999
    best_pair_z   = 0
    best_partner  = None

    # Try Non-filler partners first, then fillers
    nf_partners = [p for p in candidate_partners
                   if stype[p] != 'filler' and p != target and not is_blocked(p)]
    fl_partners = [p for p in candidate_partners
                   if stype[p] == 'filler' and p != target and not is_blocked(p)]

    for partner_pool in [nf_partners, fl_partners]:
        for partner in partner_pool:
            cfgs = pair_layer_configs(target, partner)
            for cfg in cfgs:
                z = calc_z_target_driven(cfg, target, partner)
                if z < 1: continue
                # check partner won't overshoot its own dmax
                pi     = cfg['codes'].index(partner)
                p_load = loaded[partner] + cfg['cpzs'][pi] * z
                if p_load > dmax[partner]: continue
                if cfg['gap_mm'] < best_pair_gap:
                    best_pair_gap = cfg['gap_mm']
                    best_pair     = cfg
                    best_pair_z   = z
                    best_partner  = partner
                break   # configs sorted gap ASC — first valid is best for this partner
        if best_pair is not None:
            break      # found a Non-filler partner — don't bother with fillers yet

    # If no Non-filler partner found, try fillers
    if best_pair is None:
        for partner in fl_partners:
            cfgs = pair_layer_configs(target, partner)
            for cfg in cfgs:
                z = calc_z_target_driven(cfg, target, partner)
                if z < 1: continue
                pi     = cfg['codes'].index(partner)
                p_load = loaded[partner] + cfg['cpzs'][pi] * z
                if p_load > dmax[partner]: continue
                if cfg['gap_mm'] < best_pair_gap:
                    best_pair_gap = cfg['gap_mm']
                    best_pair     = cfg
                    best_pair_z   = z
                    best_partner  = partner
                break

    # ── Decision: solo vs pair ────────────────────────────────────────────────
    use_solo = False
    if best_solo is None and best_pair is None:
        name = s_t['sku_name']
        print(f"  ❌ Cannot place {name} — no valid config (len_rem={len_rem:.0f}mm)")
        return False

    if best_pair is None:
        use_solo = True
    elif best_solo is None:
        use_solo = False
    else:
        # Prefer pair unless solo gives materially better gap
        pair_gap_excess = best_pair_gap - best_solo_gap
        use_solo = (pair_gap_excess > GAP_SOLO_PREFER_MM)

    # ── Place ─────────────────────────────────────────────────────────────────
    if use_solo:
        la, cpz, dpz_val = best_solo
        wt_pz = cpz * lookup[target]['weight_per_carton']
        cum_h = la * s_t['height_mm']
        gap   = truck['H'] - cum_h
        add_stack(
            codes=[target], names=[s_t['sku_name']],
            layers=[la], cpws=[s_t['cpw']], cpzs=[cpz],
            z=best_solo_z, cum_h=cum_h, gap=gap,
            dpz_val=dpz_val, wt_pz=wt_pz, label=label
        )
    else:
        cfg = best_pair
        add_stack(
            codes=cfg['codes'], names=cfg['names'],
            layers=cfg['layers'], cpws=cfg['cpws'], cpzs=cfg['cpzs'],
            z=best_pair_z, cum_h=cfg['cum_h'], gap=cfg['gap_mm'],
            dpz_val=cfg['dpz'], wt_pz=cfg['wt_pz'], label=label
        )
    return True

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — NON-FILLER DEMAND FULFILLMENT
# Priority: tol=0 first, then tol ASC, then planned_qty ASC (smaller first)
# ══════════════════════════════════════════════════════════════════════════════
all_codes  = list(planned.keys())
non_filler_codes = [c for c in all_codes if stype[c] != 'filler']
filler_codes     = [c for c in all_codes if stype[c] == 'filler']

def nf_sort_key(code):
    s = lookup[code]
    return (s['tolerance'], s['planned_qty'])   # tol ASC, qty ASC

non_filler_codes.sort(key=nf_sort_key)

print("="*70)
print(f"{'PHASE 1 — NON-FILLER DEMAND FULFILLMENT':^70}")
print("="*70)
print(f"  Order: {[lookup[c]['sku_name'][:20] for c in non_filler_codes]}")
print(f"  Fillers (partners/fill): {[lookup[c]['sku_name'][:20] for c in filler_codes]}")
print()

for target in non_filler_codes:
    if is_blocked(target):
        print(f"  ⏭  {lookup[target]['sku_name'][:30]} already at dmax — skip")
        continue

    # Keep placing stacks for this target until it's done or impossible
    itr = 0
    while not is_done(target) and not is_blocked(target) and len_rem > 50:
        if len(stacks) >= MAX_STACKS:
            print(f"  🛑 Max stacks reached."); break
        if wt_load >= WEIGHT_LIMIT:
            print(f"  ⚖️  Weight limit reached."); break

        # Partners = all other non-blocked SKUs (NF first, fillers second)
        partners = [c for c in all_codes if c != target and not is_blocked(c)]
        ok = place_target(target, partners, "P1-NF")
        if not ok:
            break
        itr += 1
        if itr > 20:
            print(f"  ⚠️  Loop guard hit for {lookup[target]['sku_name'][:25]}")
            break

    rem = remaining(target)
    name = lookup[target]['sku_name'][:30]
    if rem == 0 or is_done(target):
        print(f"  ✅ {name}: loaded={loaded[target]} ✓")
    else:
        print(f"  ⚠️  {name}: loaded={loaded[target]}, still need {rem} more")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 — FILLER DEMAND FULFILLMENT
# Same logic: smaller qty filler first, pairs with best available partner
# ══════════════════════════════════════════════════════════════════════════════
def fl_sort_key(code):
    return lookup[code]['planned_qty']

filler_codes.sort(key=fl_sort_key)

print()
print("="*70)
print(f"{'PHASE 2 — FILLER DEMAND FULFILLMENT':^70}")
print("="*70)

for target in filler_codes:
    if is_blocked(target):
        print(f"  ⏭  {lookup[target]['sku_name'][:30]} already at dmax — skip")
        continue

    itr = 0
    while not is_done(target) and not is_blocked(target) and len_rem > 50:
        if len(stacks) >= MAX_STACKS:
            print(f"  🛑 Max stacks reached."); break
        if wt_load >= WEIGHT_LIMIT:
            print(f"  ⚖️  Weight limit reached."); break

        partners = [c for c in all_codes if c != target and not is_blocked(c)]
        ok = place_target(target, partners, "P2-FL")
        if not ok:
            break
        itr += 1
        if itr > 20:
            print(f"  ⚠️  Loop guard hit for {lookup[target]['sku_name'][:25]}")
            break

    rem = remaining(target)
    name = lookup[target]['sku_name'][:30]
    if rem == 0 or is_done(target):
        print(f"  ✅ {name}: loaded={loaded[target]} ✓")
    else:
        print(f"  ⚠️  {name}: loaded={loaded[target]}, still need {rem} more")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 3 — FILL REMAINING TRUCK LENGTH
# Maximize cartons/mm in remaining space — all SKUs up to dmax
# ══════════════════════════════════════════════════════════════════════════════
print()
print("="*70)
print(f"{'PHASE 3 — FILL REMAINING TRUCK LENGTH':^70}")
print("="*70)

for _ in range(40):
    if len_rem < 100: break
    if len(stacks) >= MAX_STACKS: break
    if wt_load >= WEIGHT_LIMIT: break

    fillable = [c for c in all_codes if not is_blocked(c)]
    if not fillable: break

    best_cfg     = None
    best_score   = 0
    best_is_solo = False
    best_solo_t  = None

    # Evaluate all solo configs
    for code in fillable:
        for la, cpz, gap, dpz_val in solo_layer_configs(code):
            z = calc_z_fill(None, is_solo=True, solo_code=(code, la, cpz, dpz_val))
            if z < 1: break   # sorted gap ASC; if no z at best gap, deeper is worse length
            score = (cpz * z) / max(dpz_val * z, 1)   # cartons per mm
            if score > best_score:
                best_score   = score
                best_is_solo = True
                best_solo_t  = (code, la, cpz, dpz_val, gap, z)
            break   # best gap for this SKU

    # Evaluate all pair configs
    fillable_list = fillable
    for i, ca in enumerate(fillable_list):
        for cb in fillable_list[i+1:]:
            cfgs = pair_layer_configs(ca, cb)
            for cfg in cfgs:
                z = calc_z_fill(cfg)
                if z < 1: continue
                score = sum(cfg['cpzs']) * z / max(cfg['dpz'] * z, 1)
                if score > best_score:
                    best_score   = score
                    best_is_solo = False
                    best_cfg     = cfg
                    best_cfg['_z'] = z
                break   # best gap config for this pair

    if best_score == 0:
        print("  ✅ No more valid fill combos — truck fully optimised.")
        break

    if best_is_solo:
        code, la, cpz, dpz_val, gap, z = best_solo_t
        s = lookup[code]
        wt_pz = cpz * s['weight_per_carton']
        cum_h = la * s['height_mm']
        gap   = truck['H'] - cum_h
        add_stack([code], [s['sku_name']], [la], [s['cpw']], [cpz],
                  z, cum_h, gap, dpz_val, wt_pz, "P3-FILL")
    else:
        cfg = best_cfg
        z   = cfg['_z']
        add_stack(cfg['codes'], cfg['names'], cfg['layers'], cfg['cpws'], cfg['cpzs'],
                  z, cfg['cum_h'], cfg['gap_mm'], cfg['dpz'], cfg['wt_pz'], "P3-FILL")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 3 — TERMINAL RESIDUAL ROTATION RECOVERY
# Executes ONCE after the main Phase 3 fill loop ends.
# Trigger condition: len_rem > 0 but no normal-orientation stack fits.
# For each unblocked filler SKU, check whether rotating its footprint
#   (swapping depth and width) allows a single terminal stack to fit.
# This is a pure fallback micro-pass — it never replaces or re-runs
# normal-orientation logic, never touches Phase 1 or Phase 2 stacks,
# and does not globally search rotated layouts across the whole truck.
# ══════════════════════════════════════════════════════════════════════════════
print()
print("─"*70)
print(f"{'PHASE 3 ⟳ TERMINAL RESIDUAL ROTATION RECOVERY':^70}")
print("─"*70)

if len_rem >= 10 and len(stacks) < MAX_STACKS and wt_load < WEIGHT_LIMIT:
    # Only filler SKUs are candidates — they are the terminal space-fillers
    rot_candidates = [c for c in filler_codes if not is_blocked(c)]

    best_rot        = None    # (code, la, cpw_rot, cpz_rot, dpz_rot, z, cum_h, gap, wt_pz)
    best_rot_cartons = 0      # pick the rotation that recovers the most cartons

    for code in rot_candidates:
        s = lookup[code]
        # Default orientation dimensions (already chosen by best_cpw_dpz at pre-compute)
        default_dpz   = s['dpz']       # depth per zone in default orientation
        default_width = s['width_mm']  # the dimension laid across truck width (default)

        # Rotated footprint: swap depth ↔ width
        # default_dpz was the SKU dimension used as depth (length or width of box)
        # The OTHER box dimension becomes the new depth after rotation.
        # Identify which box dimensions map to default dpz and default cpw axis:
        #   best_cpw_dpz picks: opt1 → cpw=W//width_mm, dpz=length_mm
        #                       opt2 → cpw=W//length_mm, dpz=width_mm
        # So rotated_dpz = the dimension NOT currently used as dpz.
        if default_dpz == s['length_mm']:
            # Default: depth=length, width-axis=width → rotated: depth=width, width-axis=length
            rot_dpz   = s['width_mm']
            rot_cpw   = int(truck['W'] // s['length_mm'])
        else:
            # Default: depth=width, width-axis=length → rotated: depth=length, width-axis=width
            rot_dpz   = s['length_mm']
            rot_cpw   = int(truck['W'] // s['width_mm'])

        # Skip if rotation gives no improvement on depth (already the shorter dim was depth)
        if rot_dpz >= default_dpz:
            continue    # rotation does not recover residual depth — skip

        # Skip if rotated SKU doesn't fit across truck width at all
        if rot_cpw < 1:
            continue

        # Skip if rotated depth still doesn't fit in remaining length
        if rot_dpz > len_rem:
            continue

        # Layer configs are height-based — unchanged by rotation
        for la in range(1, s['maxL'] + 1):
            cum_h  = la * s['height_mm']
            if cum_h > truck['H']: break   # maxL already caps this, but guard anyway
            gap    = truck['H'] - cum_h
            cpz_rot = la * rot_cpw         # cartons per zone with rotated footprint

            if cpz_rot < 1: continue
            if not can_take_z1(code, cpz_rot): continue

            # z: how many rotated zones fit in residual depth
            z_len  = int(len_rem // rot_dpz)
            z_dmax = int((dmax[code] - loaded[code]) / cpz_rot)
            wt_pz_rot = cpz_rot * s['weight_per_carton']
            z_wt   = int((WEIGHT_LIMIT - wt_load) / wt_pz_rot) if wt_pz_rot > 0 else 9999

            z = max(0, min(z_len, z_dmax, z_wt))
            if z < 1: continue

            recovered_cartons = cpz_rot * z
            if recovered_cartons > best_rot_cartons:
                best_rot_cartons = recovered_cartons
                best_rot = (code, la, rot_cpw, cpz_rot, rot_dpz, z,
                            cum_h, gap, wt_pz_rot)
            break   # take best layer config (smallest gap first) for this SKU

    if best_rot is not None:
        code, la, rot_cpw, cpz_rot, rot_dpz, z, cum_h, gap, wt_pz_rot = best_rot
        s = lookup[code]
        print(f"  🔄 Rotation feasible for {s['sku_name'][:30]}")
        print(f"     Default dpz={s['dpz']:.0f}mm  →  Rotated dpz={rot_dpz:.0f}mm")
        print(f"     Residual len_rem={len_rem:.0f}mm  ≥  rot_dpz={rot_dpz:.0f}mm ✓")
        print(f"     Rotated cpw={rot_cpw}  cpz={cpz_rot}  z={z}  cartons={cpz_rot*z}")
        add_stack(
            codes=[code], names=[s['sku_name']],
            layers=[la], cpws=[rot_cpw], cpzs=[cpz_rot],
            z=z, cum_h=cum_h, gap=gap,
            dpz_val=rot_dpz, wt_pz=wt_pz_rot,
            label="P3-FILL", orientation="Rotated"
        )
    else:
        print("  — No rotation recovery possible (no filler SKU gains depth from rotation, ")
        print(f"    or rot_dpz > len_rem={len_rem:.0f}mm for all candidates).")
else:
    print(f"  — Skipped (len_rem={len_rem:.0f}mm / stacks={len(stacks)} / wt={wt_load:.0f}kg).")

# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
lu_tot = LENGTH_BUDGET - len_rem
print(f"\n{'='*70}")
print(f"FINAL SUMMARY")
print(f"{'='*70}")
print(f"Length : {lu_tot:.0f} / {truck['L']:.0f} mm  ({lu_tot/truck['L']*100:.1f}%)")
print(f"Weight : {wt_load:.1f} / {truck['max_wt']:.0f} kg  ({wt_load/truck['max_wt']*100:.1f}%)")
print(f"Stacks : {len(stacks)}")
print(f"Cartons: {sum(loaded.values())}")
print(f"\n{'Code':<12}{'Name':<38}{'Type':<12}{'Plan':>6}{'Min':>5}{'Max':>5}{'Load':>6}{'%':>7}  Status")
print("─"*95)
all_ok = True
for _, row in raw.iterrows():
    code = row['sku_code']; pl = int(row['planned_qty'])
    ld = loaded[code]; mn = dmin[code]; mx = dmax[code]
    pct = ld / pl * 100 if pl > 0 else 0
    if ld >= mn and ld <= mx:  flg = "✅ OK"
    elif ld > mx:              flg = "⚠️  OVER"
    else:                      flg = "❌ UNDER"; all_ok = False
    print(f"{code:<12}{row['sku_name']:<38}{row['sku_type']:<12}"
          f"{pl:>6}{mn:>5}{mx:>5}{ld:>6}{pct:>6.1f}%  {flg}")
print()
if all_ok:
    print("🎉 ALL SKUs within tolerance bounds!")
else:
    print("⚠️  Some SKUs outside bounds — check constraints above.")

# ══════════════════════════════════════════════════════════════════════════════
# BUILD DataFrames
# ══════════════════════════════════════════════════════════════════════════════
stacks_df = pd.DataFrame(stacks)
summary_rows = []
for _, row in raw.iterrows():
    code = row['sku_code']; pl = int(row['planned_qty'])
    ld = loaded[code]; mn = dmin[code]; mx = dmax[code]
    pct = ld / pl * 100 if pl > 0 else 0
    if ld >= mn and ld <= mx:  st = "✅ OK"
    elif ld > mx:              st = "⚠️ Over"
    else:                      st = "❌ Under"
    summary_rows.append({
        'sku_code': code, 'sku_name': row['sku_name'],
        'type': row['sku_type'], 'fulfillment': row['fulfillment_type'],
        'tolerance': row['tolerance'],
        'planned': pl, 'demand_min': mn, 'demand_max': mx,
        'actual': ld, 'diff': ld - pl, 'pct': round(pct, 1), 'status': st,
    })
summary_df = pd.DataFrame(summary_rows)

# ══════════════════════════════════════════════════════════════════════════════
# EXPORT EXCEL  ── v8 (fixed)
# Features:
#   ① Code A/B/C columns REMOVED
#   ② Layer cols (La/Lb/Lc) → light-yellow bold
#   ③ Z column → light-green bold
#   ④ TotCts → live =SUM(CtsA,CtsB,CtsC) formula
#   ⑤ Per-SKU loaded volume (m³) and weight (kg) columns
#   ⑥ TotVol, TotWt → live =SUM() formulas
#   ⑦ Sheet 3: Operational Analytics Summary (VFR%, Weight Util%, Length Util%)
#   ⑧ All merged-cell writes go ONLY to the top-left anchor cell — no crash
# ══════════════════════════════════════════════════════════════════════════════
def export_excel(stacks_df, summary_df, raw, truck, wt_load, len_rem,
                 fname="Loading_Plan_Output.xlsx"):

    from openpyxl.utils import get_column_letter

    # ── Safe merged-cell helper ───────────────────────────────────────────────
    # After ws.merge_cells(range), only the top-left cell is writable.
    # This helper merges and immediately returns that anchor cell object so
    # ALL property assignments happen on the same object — no MergedCell crash.
    def merge_write(ws, cell_range, value=None):
        """Merge range and return the top-left anchor cell (safe for all attrs)."""
        ws.merge_cells(cell_range)
        # Parse top-left from range string e.g. 'B2:Y2' → row=2, col=2
        top_left = cell_range.split(':')[0]   # e.g. 'B2'
        col_letter = ''.join(c for c in top_left if c.isalpha())
        row_num    = int(''.join(c for c in top_left if c.isdigit()))
        from openpyxl.utils import column_index_from_string
        col_num = column_index_from_string(col_letter)
        anchor = ws.cell(row_num, col_num)    # always returns the anchor
        if value is not None:
            anchor.value = value
        return anchor

    wb = openpyxl.Workbook()

    # ── Colour palette ────────────────────────────────────────────────────────
    NAVY  = "1F3864"; LIGHT = "D9E1F2"; GREEN = "E2EFDA"
    RED   = "FCE4D6"; AMB   = "FFF2CC"; D2    = "2E4057"
    WHITE = "FFFFFF"; TEAL  = "D6E4F0"; GOLD  = "FFD966"
    P2C   = "C5E0B4"; LYELL = "FFFACD"; LGREN = "D6F4D5"
    PURP  = "EAD7F7"; ORNG  = "FCE9D5"; STEEL = "EBF3FA"

    def _fill(h): return PatternFill("solid", fgColor=h)
    thin = Side(style='thin',   color='AAAAAA')
    med  = Side(style='medium', color='2E4057')
    def bdr():  return Border(left=thin, right=thin, top=thin, bottom=thin)
    def mbdr(): return Border(left=med,  right=med,  top=med,  bottom=med)

    TF  = Font(bold=True, color=WHITE, name='Arial', size=12)
    HF  = Font(bold=True, color=WHITE, name='Arial', size=10)
    BF  = Font(name='Arial', size=9)
    BDF = Font(bold=True,  name='Arial', size=9)
    CA  = Alignment(horizontal='center', vertical='center', wrap_text=True)
    LA  = Alignment(horizontal='left',   vertical='center', wrap_text=True)

    lu           = LENGTH_BUDGET - len_rem
    truck_vol_m3 = (truck['H'] * truck['W'] * truck['L']) / 1e9   # mm³ → m³

    # ── SKU master lookups (name → volume_m3, weight_per_carton) ─────────────
    sku_vol = dict(zip(raw['sku_name'], pd.to_numeric(raw['volume_m3'], errors='coerce').fillna(0)))
    sku_wt  = dict(zip(raw['sku_name'], pd.to_numeric(raw['weight_per_carton'], errors='coerce').fillna(0)))
    sku_type_map = dict(zip(raw['sku_name'], raw['sku_type']))

    # ══════════════════════════════════════════════════════════════════════════
    # COLUMN LAYOUT — Loading Plan (1-based column indices, no Code cols)
    #
    # B=2  Stack    C=3  Pass     D=4  #SKUs
    # E=5  Name A   F=6  La★      G=7  CpwA   H=8  CpzA   I=9  CtsA
    # J=10 VolA     K=11 WtA
    # L=12 Name B   M=13 Lb★      N=14 CpwB   O=15 CpzB   P=16 CtsB
    # Q=17 VolB     R=18 WtB
    # S=19 Name C   T=20 Lc★      U=21 CpzC   V=22 CtsC
    # W=23 VolC     X=24 WtC
    # Y=25 Z★       Z=26 H Used   AA=27 Gap   AB=28 Depth/Z
    # AC=29 Len Used  AD=30 Wt(kg)
    # AE=31 TotCts (live =SUM)
    # AF=32 TotVol (live =SUM)
    # AG=33 TotWt  (live =SUM)
    # ══════════════════════════════════════════════════════════════════════════
    COL = {
        'Stack': 2,  'Pass': 3,   'SKUs': 4,
        'NameA': 5,  'La':   6,   'CpwA': 7,  'CpzA': 8,  'CtsA': 9,
        'VolA': 10,  'WtA': 11,
        'NameB': 12, 'Lb':  13,   'CpwB':14,  'CpzB':15,  'CtsB':16,
        'VolB': 17,  'WtB': 18,
        'NameC': 19, 'Lc':  20,   'CpzC':21,  'CtsC':22,
        'VolC': 23,  'WtC': 24,
        'Z':    25,  'HUsed':26,  'Gap': 27,  'DpZ': 28,
        'LenU': 29,  'Wt':  30,
        'TotCts':31, 'TotVol':32, 'TotWt':33,
        'Orient': 34,            # AH — Normal / Rotated flag
    }
    LAST_COL = 34   # AH

    LAYER_COLS = {COL['La'], COL['Lb'], COL['Lc']}
    Z_COL      = COL['Z']
    VOL_COLS   = {COL['VolA'], COL['VolB'], COL['VolC']}
    WT_COLS    = {COL['WtA'],  COL['WtB'],  COL['WtC']}
    TOT_COLS   = {COL['TotCts'], COL['TotVol'], COL['TotWt']}

    # ══════════════════════════════════════════════════════════════════════════
    # SHEET 1 — LOADING PLAN
    # ══════════════════════════════════════════════════════════════════════════
    ws = wb.active
    ws.title = "Loading Plan"
    last_letter = get_column_letter(LAST_COL)   # AG

    # ── Title banner ──────────────────────────────────────────────────────────
    c = merge_write(ws, f'B2:{last_letter}2',
                    "NESTLÉ INDIA — TRUCK LOADING PLAN  (Optimizer v8)")
    c.font = TF; c.fill = _fill(NAVY); c.alignment = CA
    ws.row_dimensions[2].height = 30

    # ── Truck summary block ───────────────────────────────────────────────────
    c = merge_write(ws, 'B4:E4', "TRUCK SUMMARY")
    c.font = HF; c.fill = _fill(NAVY); c.alignment = CA

    params = [
        ("H × W × L (mm)",   f"{truck['H']} × {truck['W']} × {truck['L']}",   None),
        ("Max Weight (kg)",   truck['max_wt'],                                   None),
        ("Truck Volume (m³)", f"{truck_vol_m3:.4f}",                             STEEL),
        ("Length Used",       f"{lu:.0f} mm  ({lu/truck['L']*100:.1f}%)",        None),
        ("Length Free",       f"{len_rem:.0f} mm",                               None),
        ("Weight Loaded",     f"{wt_load:.1f} kg  ({wt_load/truck['max_wt']*100:.1f}%)", None),
        ("Active Stacks",     len(stacks_df),                                    None),
        ("Total Cartons",     int(stacks_df['total_cartons'].sum()) if len(stacks_df) > 0 else 0, None),
    ]
    for r_off, (lbl, val, override_fill) in enumerate(params):
        r = 5 + r_off
        lc = ws.cell(r, 2, lbl)
        lc.font = BDF; lc.fill = _fill(LIGHT); lc.border = bdr()
        vc = ws.cell(r, 3, val)
        vc.font = BF; vc.alignment = CA; vc.border = bdr()
        if override_fill:
            vc.fill = _fill(override_fill)
        elif "Weight Loaded" in lbl:
            vc.fill = _fill(RED) if wt_load > truck['max_wt'] else _fill(GREEN)
        elif "Length Used" in lbl:
            vc.fill = _fill(RED) if lu > truck['L'] else _fill(GREEN)

    # ── Sub-header bar ────────────────────────────────────────────────────────
    ROW = 15   # column header row (one extra row because of Truck Volume param)
    c = merge_write(ws, f'B{ROW-1}:{last_letter}{ROW-1}',
        "STACK CONFIGURATIONS — LOADER INSTRUCTION CARD  "
        "| 🔵 Blue=P1 Non-filler  | 🟢 Green=P2 Filler  | 🩵 Teal=P3 Fill remaining  | 🔄 Amber Orient=Rotated")
    c.font = HF; c.fill = _fill(NAVY); c.alignment = CA

    # ── Column headers ────────────────────────────────────────────────────────
    heads = [
        (COL['Stack'],  "Stack",          D2,    None),
        (COL['Pass'],   "Pass",           D2,    None),
        (COL['SKUs'],   "# SKUs",         D2,    None),
        (COL['NameA'],  "Name A",         D2,    None),
        (COL['La'],     "La",             LYELL, Font(bold=True, name='Arial', size=10, color='7B4F00')),
        (COL['CpwA'],   "CpwA",           D2,    None),
        (COL['CpzA'],   "CpzA",           D2,    None),
        (COL['CtsA'],   "CtsA",           D2,    None),
        (COL['VolA'],   "Vol A\n(m³)",    PURP,  Font(bold=True, name='Arial', size=9, color='FFFFFF')),
        (COL['WtA'],    "Wt A\n(kg)",     PURP,  Font(bold=True, name='Arial', size=9, color='FFFFFF')),
        (COL['NameB'],  "Name B",         D2,    None),
        (COL['Lb'],     "Lb",             LYELL, Font(bold=True, name='Arial', size=10, color='7B4F00')),
        (COL['CpwB'],   "CpwB",           D2,    None),
        (COL['CpzB'],   "CpzB",           D2,    None),
        (COL['CtsB'],   "CtsB",           D2,    None),
        (COL['VolB'],   "Vol B\n(m³)",    PURP,  Font(bold=True, name='Arial', size=9, color='FFFFFF')),
        (COL['WtB'],    "Wt B\n(kg)",     PURP,  Font(bold=True, name='Arial', size=9, color='FFFFFF')),
        (COL['NameC'],  "Name C",         D2,    None),
        (COL['Lc'],     "Lc",             LYELL, Font(bold=True, name='Arial', size=10, color='7B4F00')),
        (COL['CpzC'],   "CpzC",           D2,    None),
        (COL['CtsC'],   "CtsC",           D2,    None),
        (COL['VolC'],   "Vol C\n(m³)",    PURP,  Font(bold=True, name='Arial', size=9, color='FFFFFF')),
        (COL['WtC'],    "Wt C\n(kg)",     PURP,  Font(bold=True, name='Arial', size=9, color='FFFFFF')),
        (COL['Z'],      "Z",              LGREN, Font(bold=True, name='Arial', size=10, color='1A5C1A')),
        (COL['HUsed'],  "H Used\n(mm)",   D2,    None),
        (COL['Gap'],    "Gap\n(mm)",      D2,    None),
        (COL['DpZ'],    "Depth/Z\n(mm)", D2,    None),
        (COL['LenU'],   "Len Used\n(mm)", D2,    None),
        (COL['Wt'],     "Stack\nWt(kg)", D2,    None),
        (COL['TotCts'], "Total\nCts",    GOLD,  Font(bold=True, name='Arial', size=10, color='3D2B00')),
        (COL['TotVol'], "Total\nVol(m³)",GOLD,  Font(bold=True, name='Arial', size=10, color='3D2B00')),
        (COL['TotWt'],  "Total\nWt(kg)", GOLD,  Font(bold=True, name='Arial', size=10, color='3D2B00')),
        (COL['Orient'], "Orient\nFlag",   AMB,   Font(bold=True, name='Arial', size=9,  color='5C3A00')),
    ]
    for ci, hdr, bg, fnt in heads:
        cell = ws.cell(ROW, ci, hdr)
        cell.fill = _fill(bg); cell.border = bdr(); cell.alignment = CA
        cell.font = fnt if fnt else HF
    ws.row_dimensions[ROW].height = 38

    # ── Pass fill map ─────────────────────────────────────────────────────────
    PASS_FILLS = {'P1-NF': _fill(LIGHT), 'P2-FL': _fill(P2C), 'P3-FILL': _fill(TEAL)}

    # ── Data rows ─────────────────────────────────────────────────────────────
    DATA_START = ROW + 1
    for r, (_, st) in enumerate(stacks_df.iterrows()):
        rn       = DATA_START + r
        row_fill = PASS_FILLS.get(st.get('pass', 'P1-NF'), _fill(LIGHT))

        name0 = st.get('name_0', '') or ''
        name1 = st.get('name_1', '') or ''
        name2 = st.get('name_2', '') or ''
        cts0  = int(st.get('cartons_0', 0) or 0)
        cts1  = int(st.get('cartons_1', 0) or 0)
        cts2  = int(st.get('cartons_2', 0) or 0)
        lv0   = round((sku_vol.get(name0, 0) or 0) * cts0, 4)
        lv1   = round((sku_vol.get(name1, 0) or 0) * cts1, 4)
        lv2   = round((sku_vol.get(name2, 0) or 0) * cts2, 4)
        lw0   = round((sku_wt.get(name0,  0) or 0) * cts0, 1)
        lw1   = round((sku_wt.get(name1,  0) or 0) * cts1, 1)
        lw2   = round((sku_wt.get(name2,  0) or 0) * cts2, 1)

        # All non-formula cells
        data = [
            (COL['Stack'],  st['stack_id']),
            (COL['Pass'],   st.get('pass', '')),
            (COL['SKUs'],   st.get('n_skus', 1)),
            (COL['NameA'],  name0),
            (COL['La'],     st.get('layers_0', 0)),
            (COL['CpwA'],   st.get('cpw_0', 0)),
            (COL['CpzA'],   st.get('cpz_0', 0)),
            (COL['CtsA'],   cts0),
            (COL['VolA'],   lv0),
            (COL['WtA'],    lw0),
            (COL['NameB'],  name1),
            (COL['Lb'],     st.get('layers_1', 0)),
            (COL['CpwB'],   st.get('cpw_1', 0)),
            (COL['CpzB'],   st.get('cpz_1', 0)),
            (COL['CtsB'],   cts1),
            (COL['VolB'],   lv1),
            (COL['WtB'],    lw1),
            (COL['NameC'],  name2),
            (COL['Lc'],     st.get('layers_2', 0)),
            (COL['CpzC'],   st.get('cpz_2', 0)),
            (COL['CtsC'],   cts2),
            (COL['VolC'],   lv2),
            (COL['WtC'],    lw2),
            (COL['Z'],      st['z']),
            (COL['HUsed'],  round(st['cum_h'])),
            (COL['Gap'],    round(st['gap_mm'])),
            (COL['DpZ'],    round(st['dpz'])),
            (COL['LenU'],   round(st['length_used'])),
            (COL['Wt'],     st['wt_kg']),
            (COL['Orient'], st.get('orientation', 'Normal')),
        ]
        for ci, val in data:
            cell = ws.cell(rn, ci, val)
            cell.border = bdr(); cell.alignment = CA
            if ci in LAYER_COLS:
                cell.fill = _fill(LYELL)
                cell.font = Font(bold=True, name='Arial', size=9)
            elif ci == Z_COL:
                cell.fill = _fill(LGREN)
                cell.font = Font(bold=True, name='Arial', size=9)
            elif ci in VOL_COLS:
                cell.fill = _fill(PURP)
                cell.font = Font(italic=True, name='Arial', size=9, color='3B1F6B')
                cell.number_format = '0.0000'
            elif ci in WT_COLS:
                cell.fill = _fill(ORNG)
                cell.font = Font(italic=True, name='Arial', size=9, color='5C2D00')
                cell.number_format = '0.0'
            elif ci == COL['Orient']:
                # Rotated stacks get a warm amber highlight; Normal stays neutral
                is_rotated = (val == 'Rotated')
                cell.fill = _fill('F4B942') if is_rotated else _fill(AMB)
                cell.font = Font(bold=is_rotated, name='Arial', size=9,
                                 color='5C3A00' if is_rotated else '666666')
            else:
                cell.fill = row_fill; cell.font = BF

        # Gap conditional colour
        gc = ws.cell(rn, COL['Gap'])
        gap_v = st['gap_mm']
        gc.fill = _fill(GREEN) if gap_v < 50 else (_fill(AMB) if gap_v < 150 else _fill(RED))
        gc.font = BF

        # ── Live formulas — no merges involved, safe to write directly ────────
        ca  = get_column_letter(COL['CtsA']); cb = get_column_letter(COL['CtsB'])
        cc  = get_column_letter(COL['CtsC'])
        va  = get_column_letter(COL['VolA']); vb = get_column_letter(COL['VolB'])
        vc  = get_column_letter(COL['VolC'])
        wa  = get_column_letter(COL['WtA']);  wb2= get_column_letter(COL['WtB'])
        wc  = get_column_letter(COL['WtC'])

        for col_k, formula, fmt in [
            ('TotCts', f"=SUM({ca}{rn},{cb}{rn},{cc}{rn})", None),
            ('TotVol', f"=SUM({va}{rn},{vb}{rn},{vc}{rn})", '0.0000'),
            ('TotWt',  f"=SUM({wa}{rn},{wb2}{rn},{wc}{rn})", '0.0'),
        ]:
            fc = ws.cell(rn, COL[col_k], formula)
            fc.fill = _fill(GOLD); fc.border = bdr(); fc.alignment = CA
            fc.font = Font(bold=True, name='Arial', size=9)
            if fmt: fc.number_format = fmt

    # ── Column widths ─────────────────────────────────────────────────────────
    col_widths = {
        COL['Stack']: 7,  COL['Pass']: 8,   COL['SKUs']: 6,
        COL['NameA']: 30, COL['La']:   5,   COL['CpwA']: 6, COL['CpzA']: 6,
        COL['CtsA']:  7,  COL['VolA']: 9,   COL['WtA']:  8,
        COL['NameB']: 30, COL['Lb']:   5,   COL['CpwB']: 6, COL['CpzB']: 6,
        COL['CtsB']:  7,  COL['VolB']: 9,   COL['WtB']:  8,
        COL['NameC']: 30, COL['Lc']:   5,   COL['CpzC']: 6,
        COL['CtsC']:  7,  COL['VolC']: 9,   COL['WtC']:  8,
        COL['Z']:     5,  COL['HUsed']:9,   COL['Gap']:  8, COL['DpZ']:  9,
        COL['LenU']: 10,  COL['Wt']:   9,
        COL['TotCts']:10, COL['TotVol']:11, COL['TotWt']:10,
        COL['Orient']: 10,
    }
    for ci, w in col_widths.items():
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.freeze_panes = f"E{ROW+1}"

    # ══════════════════════════════════════════════════════════════════════════
    # SHEET 2 — SKU DEMAND FULFILLMENT SUMMARY
    # ══════════════════════════════════════════════════════════════════════════
    ws2 = wb.create_sheet("SKU Summary")

    c = merge_write(ws2, 'B2:N2', "SKU DEMAND FULFILLMENT SUMMARY")
    c.font = TF; c.fill = _fill(NAVY); c.alignment = CA
    ws2.row_dimensions[2].height = 25

    sum_hdrs = ["Code","SKU Name","Type","Fulfillment","Tolerance",
                "Planned","Min","Max","Actual","Diff","% Met","Status",
                "Vol/Ctn\n(m³)","Wt/Ctn\n(kg)"]
    for col_i, h in enumerate(sum_hdrs, 2):
        cell = ws2.cell(4, col_i, h)
        cell.font = HF; cell.fill = _fill(NAVY); cell.alignment = CA; cell.border = bdr()
    ws2.row_dimensions[4].height = 28

    for r, row in summary_df.iterrows():
        rn = 5 + r
        vol_pc = sku_vol.get(row['sku_name'], 0) or 0
        wt_pc  = sku_wt.get(row['sku_name'], 0) or 0
        vals = [
            row['sku_code'], row['sku_name'], row['type'], row['fulfillment'],
            f"{row['tolerance']*100:.0f}%",
            row['planned'], row['demand_min'], row['demand_max'],
            row['actual'], row['diff'], f"{row['pct']}%", row['status'],
            round(vol_pc, 4), round(wt_pc, 2),
        ]
        for col_i, v in enumerate(vals, 2):
            cell = ws2.cell(rn, col_i, v)
            cell.font = BF; cell.border = bdr(); cell.alignment = CA
        status = row['status']
        ws2.cell(rn, 13).fill = (
            _fill(GREEN) if status == "✅ OK"
            else _fill(AMB)  if status == "⚠️ Over"
            else _fill(RED)
        )
        ws2.cell(rn, 14).fill = _fill(PURP)
        ws2.cell(rn, 15).fill = _fill(ORNG)

    for col, w in zip(['B','C','D','E','F','G','H','I','J','K','L','M','N','O'],
                      [12, 40, 12, 12, 9, 9, 9, 9, 9, 7, 9, 14, 13, 13]):
        ws2.column_dimensions[col].width = w
    ws2.freeze_panes = 'B5'

    # ══════════════════════════════════════════════════════════════════════════
    # SHEET 3 — OPERATIONAL ANALYTICS SUMMARY
    # Layout rules to avoid MergedCell crash:
    #   • LEFT KPI card  uses columns B:E  (cols 2-5)
    #   • RIGHT KPI card uses columns G:J  (cols 7-10)
    #   • Col F (col 6) is a blank spacer — never merged or written into
    #   • The two column ranges are COMPLETELY non-overlapping → no crash
    #   • merge_write() ensures value+formatting go only to the anchor cell
    # ══════════════════════════════════════════════════════════════════════════
    ws3 = wb.create_sheet("Analytics Summary")

    c = merge_write(ws3, 'B2:J2',
                    "VFR OPERATIONAL ANALYTICS SUMMARY — NESTLÉ INDIA")
    c.font = TF; c.fill = _fill(NAVY); c.alignment = CA
    ws3.row_dimensions[2].height = 32

    c = merge_write(ws3, 'B3:J3',
                    f"Truck: {truck['H']}mm H × {truck['W']}mm W × {truck['L']}mm L   "
                    f"|   Optimizer v8   |   {len(stacks_df)} stacks")
    c.font = Font(italic=True, name='Arial', size=10, color='555555')
    c.alignment = CA
    ws3.row_dimensions[3].height = 18

    # ── Pre-compute all KPI values ────────────────────────────────────────────
    total_loaded_vol = 0.0
    for _, st in stacks_df.iterrows():
        for slot in range(3):
            n = st.get(f'name_{slot}', '') or ''
            c_val = int(st.get(f'cartons_{slot}', 0) or 0)
            total_loaded_vol += (sku_vol.get(n, 0) or 0) * c_val
    total_loaded_vol = round(total_loaded_vol, 4)
    total_loaded_wt  = round(wt_load, 1)
    total_loaded_cts = int(stacks_df['total_cartons'].sum()) if len(stacks_df) > 0 else 0
    realized_vfr_pct = round((total_loaded_vol / truck_vol_m3 * 100) if truck_vol_m3 > 0 else 0, 2)
    wt_util_pct      = round(total_loaded_wt / truck['max_wt'] * 100, 1)
    len_util_pct     = round(lu / truck['L'] * 100, 1)

    # ── KPI pair definitions: (left_label, left_value, left_unit,
    #                           right_label, right_value, right_unit,
    #                           left_fill, right_fill) ──────────────────────
    kpi_pairs = [
        ("📦 Truck Volume",        f"{truck_vol_m3:.4f} m³",   "Usable cargo space",
         "⚖️ Max Weight Cap.",      f"{truck['max_wt']:.0f} kg", "Truck capacity",
         STEEL, ORNG),
        ("📊 Total Loaded Volume", f"{total_loaded_vol:.4f} m³","Sum of (SKU vol × cartons)",
         "⚖️ Total Loaded Weight", f"{total_loaded_wt:.1f} kg", "Sum of (SKU wt × cartons)",
         PURP, ORNG),
        ("🎯 Realized VFR %",      f"{realized_vfr_pct:.2f}%", "Loaded Vol / Truck Vol",
         "📈 Weight Utilization",  f"{wt_util_pct:.1f}%",       "Loaded Wt / Max Wt",
         LGREN, AMB),
        ("📏 Length Utilization",  f"{len_util_pct:.1f}%",      f"{lu:.0f} mm of {truck['L']:.0f} mm used",
         "🗂️ Total Cartons",       f"{total_loaded_cts}",        "Across all stacks",
         LIGHT, GOLD),
        ("📏 Length Used",         f"{lu:.0f} mm",              "Truck length consumed",
         "🔢 Active Stacks",       f"{len(stacks_df)}",          "Stack configs placed",
         LIGHT, LIGHT),
    ]

    KPI_LBL_FONT = Font(bold=True,  name='Arial', size=10, color='1F3864')
    KPI_VAL_FONT = Font(bold=True,  name='Arial', size=14, color='2E4057')
    KPI_UNT_FONT = Font(italic=True,name='Arial', size=9,  color='888888')

    START_R = 5
    for i, (ll, lv, lu_txt, rl, rv, ru_txt, lfill, rfill) in enumerate(kpi_pairs):
        base = START_R + i * 4   # 4 rows per KPI pair (label, value, unit, spacer)

        # LEFT card — merged B:E  (cols 2-5) — 3 content rows
        for row_off, (txt, fnt, fill_c, ht) in enumerate([
            (ll, KPI_LBL_FONT, lfill, 16),
            (lv, KPI_VAL_FONT, lfill, 22),
            (lu_txt, KPI_UNT_FONT, lfill, 13),
        ]):
            r = base + row_off
            anchor = merge_write(ws3, f'B{r}:E{r}', txt)
            anchor.font = fnt
            anchor.fill = _fill(fill_c)
            anchor.alignment = LA
            anchor.border = mbdr() if row_off == 0 else bdr()
            ws3.row_dimensions[r].height = ht

        # RIGHT card — merged G:J  (cols 7-10) — non-overlapping with B:E
        for row_off, (txt, fnt, fill_c, ht) in enumerate([
            (rl, KPI_LBL_FONT, rfill, 16),
            (rv, KPI_VAL_FONT, rfill, 22),
            (ru_txt, KPI_UNT_FONT, rfill, 13),
        ]):
            r = base + row_off
            anchor = merge_write(ws3, f'G{r}:J{r}', txt)
            anchor.font = fnt
            anchor.fill = _fill(fill_c)
            anchor.alignment = LA
            anchor.border = mbdr() if row_off == 0 else bdr()

    # ── SKU Contribution Table ────────────────────────────────────────────────
    ctrib_start = START_R + len(kpi_pairs) * 4 + 1

    c = merge_write(ws3, f'B{ctrib_start}:J{ctrib_start}',
                    "SKU VOLUME & WEIGHT CONTRIBUTION")
    c.font = HF; c.fill = _fill(NAVY); c.alignment = CA
    ws3.row_dimensions[ctrib_start].height = 22

    ctrib_hdrs = ["SKU Name","Type","Loaded\nCartons","Vol/Ctn\n(m³)",
                  "Total\nVol(m³)","Wt/Ctn\n(kg)","Total\nWt(kg)","Status"]
    for col_i, h in enumerate(ctrib_hdrs, 2):
        cell = ws3.cell(ctrib_start + 1, col_i, h)
        cell.font = HF; cell.fill = _fill(D2); cell.alignment = CA; cell.border = bdr()
    ws3.row_dimensions[ctrib_start + 1].height = 32

    # Aggregate loaded cartons per SKU name across all stacks
    sku_agg = {}
    for _, st in stacks_df.iterrows():
        for slot in range(3):
            name = st.get(f'name_{slot}', '') or ''
            cts  = int(st.get(f'cartons_{slot}', 0) or 0)
            if not name or cts == 0: continue
            sku_agg[name] = sku_agg.get(name, 0) + cts

    row_fills_cycle = [_fill(LIGHT), _fill(STEEL)]
    for ri, (name, total_cts) in enumerate(sku_agg.items()):
        r = ctrib_start + 2 + ri
        vol_pc   = sku_vol.get(name, 0) or 0
        wt_pc    = sku_wt.get(name, 0) or 0
        tot_vol  = round(vol_pc * total_cts, 4)
        tot_wt   = round(wt_pc  * total_cts, 1)
        stype    = sku_type_map.get(name, '')
        status   = "✅ filler" if 'filler' in stype else "📦 non-filler"
        row_f    = _fill(P2C) if 'filler' in stype else row_fills_cycle[ri % 2]

        vals = [name, stype, total_cts, round(vol_pc,4), tot_vol, round(wt_pc,2), tot_wt, status]
        for col_i, v in enumerate(vals, 2):
            cell = ws3.cell(r, col_i, v)
            cell.font = BF; cell.border = bdr(); cell.alignment = CA; cell.fill = row_f
        ws3.cell(r, 6).fill  = _fill(PURP)   # Total Vol tint
        ws3.cell(r, 8).fill  = _fill(ORNG)   # Total Wt tint
        ws3.row_dimensions[r].height = 16

    # Grand total row
    grand_r = ctrib_start + 2 + len(sku_agg)
    grand_vals = ["TOTAL", "", total_loaded_cts, "",
                  round(total_loaded_vol,4), "", round(total_loaded_wt,1), ""]
    for col_i, v in enumerate(grand_vals, 2):
        cell = ws3.cell(grand_r, col_i, v)
        cell.font = Font(bold=True, name='Arial', size=10)
        cell.fill = _fill(GOLD); cell.border = mbdr(); cell.alignment = CA
    ws3.row_dimensions[grand_r].height = 20

    # Column widths for Sheet 3
    for col, w in zip(['B','C','D','E','F','G','H','I','J'],
                      [40, 14, 12, 14, 4, 40, 12, 14, 14]):
        ws3.column_dimensions[col].width = w
    ws3.freeze_panes = 'B5'

    # ── Save ──────────────────────────────────────────────────────────────────
    wb.save(fname)
    print(f"\n✅ Saved: {fname}")
    return fname


# ── Generate timestamped output filename and save to OUTPUT/ ──────────────────
timestamp   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
output_stem = f"Loading_Plan_Output_{timestamp}.xlsx"
output_path = OUTPUT_DIR / output_stem

output_file = export_excel(stacks_df, summary_df, raw, truck, wt_load, len_rem,
                            fname=str(output_path))

print(f"\n📥 Output saved → {output_path}")
print("\n📊 v9 Export — 3 Sheets:")
print("  Sheet 1 — Loading Plan   : stacks + Vol/Wt columns + live TotCts/TotVol/TotWt")
print("  Sheet 2 — SKU Summary    : demand fulfillment + Vol/Wt per carton reference")
print("  Sheet 3 — Analytics      : VFR%, Wt Util%, Len Util%, SKU contribution table")
print("\n🎨 Highlights:")
print("  🟡 Yellow+Bold  = Layer cols (La, Lb, Lc)")
print("  🟢 Green+Bold   = Z column")
print("  🟣 Purple       = Loaded volume contribution (VolA/B/C)")
print("  🟠 Orange       = Loaded weight contribution (WtA/B/C)")
print("  🟨 Gold formula = TotCts / TotVol / TotWt (live =SUM)")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Validate output + finish up
# ─────────────────────────────────────────────────────────────────────────────
print("Generating output...")

try:
    if 'output_file' not in globals():
        raise RuntimeError("Optimizer did not produce an output reference.")

    out_path = Path(output_file)

    if not out_path.exists() or out_path.stat().st_size == 0:
        raise RuntimeError(f"Output file was not created or is empty: {out_path}")

    # Required-sheets check (matches the 3 sheets built by export_excel)
    wb_check = openpyxl.load_workbook(out_path, read_only=True)
    required_sheets = {"Loading Plan", "SKU Summary", "Analytics Summary"}
    missing = required_sheets - set(wb_check.sheetnames)
    wb_check.close()
    if missing:
        raise RuntimeError(f"Output workbook is missing expected sheet(s): {missing}")

    # Sanity check on the underlying dataframe, if available
    if 'stacks_df' in globals() and stacks_df is not None and len(stacks_df) == 0:
        log("Warning: stacks_df is empty — output has no loaded stacks.", level="WARNING")
        print("⚠️  Warning: no stacks were loaded — please check input data/tolerances.")

    elapsed = (datetime.datetime.now() - _run_started).total_seconds()
    log(f"Output validated OK: {out_path.name} ({out_path.stat().st_size} bytes)")
    log(f"Run completed successfully in {elapsed:.1f}s. Output: {out_path}")

    print(f"\n✅ Done. Output ready at:")
    print(f"   {out_path}")
    print(f"⏱️  Total run time: {elapsed:.1f} seconds")
    print(f"📝 Full log: {LOG_FILE}")

except Exception as e:
    log(f"Output validation FAILED: {e}", level="ERROR")
    raise RuntimeError(
        "The optimizer ran, but the output file could not be validated.\n"
        f"   → Technical detail: {e}\n"
        f"   → Check this file for the full run history: {LOG_FILE}"
    )
